# 03 — Sampling, training, classification, temporal filtering, validation

This notebook runs the deployed mapping pipeline: NDVI-guided point sampling from reference polygons, feature extraction at points, merging of the two mosaics' training tables, Random Forest training, per-mosaic classification, mosaic merging into the annual map, temporal filtering to the permanent layer, and map validation.

> **Where the reported metrics come from.** Feature selection is performed in `04_feature_selection_leakage_free.ipynb` and the validation metrics in `05_spatial_validation.ipynb`. The Random Forest here is trained on the selected 30-feature set to generate the deployed yearly rasters; its in-notebook fit is the deployment model, not the source of the reported accuracy.

Set `DATA_DIR` to your project root. The two mosaics are processed by the same code by pointing the stack/path variables at mosaic 1 or mosaic 2.

_Manuscript: Methods — Sampling, classification, temporal filtering, validation._

### NDVI extraction at reference polygons

In [ ]:
#import os
#from pyproj import datadir

#os.environ["PROJ_LIB"] = datadir.get_data_dir()
import geopandas as gpd
import rasterio
from rasterio.mask import mask


# 📌 Define file paths
gpkg_path = r"DATA_DIR\Combined\2024\Vectors\pinus_mugo_2024_refpolys_updated20250418.gpkg"  # GPKG with polygons
ndvi_raster_path = r"DATA_DIR\NDVI_comp_2024_full.tif"  # NDVI raster
output_presence = r"DATA_DIR\Combined\2024\Rasters\NDVI_presence.tif"  # Clipped presence raster
output_absence = r"DATA_DIR\Combined\2024\Rasters\NDVI_absence.tif"  # Clipped absence raster

# 📌 Step 1: Load Polygons from GPKG
gdf = gpd.read_file(gpkg_path)

# Ensure the column 'class' exists
if 'class' not in gdf.columns:
    raise ValueError("❌ The GPKG file does not have a 'class' column!")

# Separate presence (1) and absence (0) polygons
presence_polygons = gdf[gdf['class'] == 1]["geometry"]
absence_polygons = gdf[gdf['class'] == 0]["geometry"]

# Convert geometries to JSON format for rasterio.mask
presence_shapes = [geom.__geo_interface__ for geom in presence_polygons]
absence_shapes = [geom.__geo_interface__ for geom in absence_polygons]

# 📌 Step 2: Open NDVI Raster & Clip
with rasterio.open(ndvi_raster_path) as src:
    meta = src.meta.copy()

    # Clip for presence (class 1)
    if presence_shapes:
        presence_data, presence_transform = mask(src, presence_shapes, crop=True)
        meta.update({"transform": presence_transform, "height": presence_data.shape[1], "width": presence_data.shape[2]})
        with rasterio.open(output_presence, "w", **meta) as dst:
            dst.write(presence_data)
        print(f"✅ Clipped NDVI raster for presence saved at: {output_presence}")

    # Clip for absence (class 0)
    if absence_shapes:
        absence_data, absence_transform = mask(src, absence_shapes, crop=True)
        meta.update({"transform": absence_transform, "height": absence_data.shape[1], "width": absence_data.shape[2]})
        with rasterio.open(output_absence, "w", **meta) as dst:
            dst.write(absence_data)
        print(f"✅ Clipped NDVI raster for absence saved at: {output_absence}")


### NDVI-threshold + spatial-thinning point sampling

In [ ]:
import geopandas as gpd
import rasterio
import numpy as np
import pandas as pd
from shapely.geometry import Point
from scipy.spatial import cKDTree

# 📌 Define File Paths
presence_raster_path = r"DATA_DIR\Combined\2024\Rasters\NDVI_presence.tif"
absence_raster_path = r"DATA_DIR\Combined\2024\Rasters\NDVI_absence.tif"
output_shapefile = r"DATA_DIR\Combined\2024\Vectors\training_points_combined.shp"

# 📌 Function to Extract Points from Raster
def extract_points(raster_path, ndvi_threshold, min_distance_pixels, presence_type):
    """
    Extracts valid pixels above a given NDVI threshold and applies spatial thinning.
    - raster_path: Path to the clipped NDVI raster.
    - ndvi_threshold: Minimum NDVI value to be considered.
    - min_distance_pixels: Minimum pixel separation for thinning.
    - presence_type: 1 for presence, 0 for absence.
    """
    with rasterio.open(raster_path) as src:
        raster_data = src.read(1)  # Read the single-band raster
        raster_transform = src.transform
        raster_crs = src.crs
        nodata_value = src.nodata
        pixel_size = src.res[0]  # Get pixel resolution in meters

        # Identify valid pixels based on threshold
        rows, cols = np.where((raster_data > ndvi_threshold) & (raster_data != nodata_value))

        # Convert pixel indices to real-world coordinates
        def pixel_to_coords(row, col, transform):
            x, y = rasterio.transform.xy(transform, row, col, offset="center")
            return x, y

        pixel_coords = [pixel_to_coords(r, c, raster_transform) for r, c in zip(rows, cols)]
        points = [Point(x, y) for x, y in pixel_coords]

        # Apply spatial thinning (min distance in meters)
        min_distance_meters = min_distance_pixels * pixel_size

        def spatial_thinning(points, min_distance):
            if len(points) == 0:
                return []
            coords = np.array([(p.x, p.y) for p in points])
            tree = cKDTree(coords)
            mask_keep = np.ones(len(coords), dtype=bool)
            selected_indices = []

            for i in range(len(coords)):
                if mask_keep[i]:  
                    selected_indices.append(i)  
                    neighbors = tree.query_ball_point(coords[i], min_distance)  
                    mask_keep[neighbors] = False  
                    mask_keep[i] = True  

            return [points[i] for i in selected_indices]

        # Apply thinning
        selected_points = spatial_thinning(points, min_distance_meters)

        # Create GeoDataFrame
        gdf = gpd.GeoDataFrame(geometry=selected_points, crs=raster_crs)
        gdf["type"] = presence_type  # Assign class label (1 = presence, 0 = absence)

        return gdf

# 📌 Extract Presence & Absence Points
presence_gdf = extract_points(presence_raster_path, ndvi_threshold=0.5, min_distance_pixels=3, presence_type=1)
absence_gdf = extract_points(absence_raster_path, ndvi_threshold=0.0, min_distance_pixels=10, presence_type=0)

# 📌 Merge and Save Final Training Dataset
final_training_gdf = gpd.GeoDataFrame(pd.concat([presence_gdf, absence_gdf], ignore_index=True))
final_training_gdf.to_file(output_shapefile)

print(f"✅ Final training points saved: {output_shapefile}")
print(final_training_gdf.head())  # Display first few rows


### Align & resample rasters into the feature stack

In [ ]:
import os
import glob
import rasterio
import numpy as np
from rasterio.enums import Resampling
from rasterio.warp import reproject, calculate_default_transform

# 📌 Define Paths
raster_folder = r"DATA_DIR\Combined\2024\Rasters\Mosaic2\selected"  # Folder with rasters
output_stack = r"DATA_DIR\Combined\2024\Rasters\Mosaic2\raster_stack_S2_selected30_mosaic2.tif"  # Output stacked raster
reference_raster_path = r"DATA_DIR\Combined\2024\Rasters\Mosaic2\selected\DTM_resampled.tif"  # Reference raster

# 📌 Step 1: Load Reference Raster
with rasterio.open(reference_raster_path) as ref:
    ref_meta = ref.meta.copy()
    ref_transform = ref.transform
    ref_crs = ref.crs
    ref_width, ref_height = ref.width, ref.height
    ref_res = ref.res  # Get pixel resolution
    ref_bounds = ref.bounds  # Get extent

# 📌 Step 2: Get All Raster Files
raster_files = glob.glob(os.path.join(raster_folder, "*.tif"))

# Lists to Store Processed Data
all_bands = []
band_names = []

# 📌 Step 3: Process Each Raster File
for raster_path in raster_files:
    print(f"📂 Processing: {raster_path}")

    with rasterio.open(raster_path) as src:
        if src.crs is None:
            print(f"  ⚠️ WARNING: {base_name} has no CRS. Assigning reference CRS.")
            src_crs = ref_crs
        else:
            src_crs = src.crs
        base_name = os.path.basename(raster_path).replace(".tif", "")  # Extract base name
        num_bands = src.count  # Get number of bands

        # Check if raster already matches reference in resolution, extent, and grid
        same_extent = src.bounds == ref_bounds
        same_res = np.allclose(src.res, ref_res, atol=1e-6)  # Compare pixel sizes

        for band_idx in range(1, num_bands + 1):
            print(f"  📊 Reading Band {band_idx}/{num_bands} from {base_name}")

            # Read the band
            data = src.read(band_idx)

            # If necessary, reproject and align the raster
            if not (same_extent and same_res):
                print(f"  🔄 Aligning & Resampling Band {band_idx} to match reference raster...")
                
                # Compute transformation for reprojection
                transform, width, height = calculate_default_transform(
                    src.crs, ref_crs, ref_width, ref_height, *ref_bounds
                )

                # Create an empty array to store reprojected data
                aligned_data = np.empty((ref_height, ref_width), dtype=np.float32)

                # Reproject the raster to match the reference raster's extent & resolution
                reproject(
                    source=data,
                    destination=aligned_data,
                    src_transform=src.transform,
                    src_crs=src_crs,  # Use fixed version,
                    dst_transform=transform,
                    dst_crs=ref_crs,
                    resampling=Resampling.bilinear
                )
            else:
                aligned_data = data  # Use the original data if it already matches

            all_bands.append(aligned_data.astype(np.float32))  # Store the processed band
            band_names.append(f"{base_name}_B{band_idx}" if num_bands > 1 else base_name)


# 📌 Step 4: Stack Bands & Save Output Raster
stacked_array = np.stack(all_bands, axis=0)  # Stack into multi-band array

# Update metadata for multi-band output
ref_meta.update({
    "count": stacked_array.shape[0],  # Number of total bands
    "dtype": "float32",
    "compress": "lzw",  # Compression for efficiency
    "BIGTIFF": "YES",  # Enable BigTIFF for large files
    "tiled": True,  # Optimize writing speed
})

# Save the final raster stack
with rasterio.open(output_stack, "w", **ref_meta) as dst:
    dst.write(stacked_array)

    # Assign band names for QGIS & ArcGIS compatibility
    for idx, band_name in enumerate(band_names):
        dst.set_band_description(idx + 1, band_name)

print(f"✅ Raster stack saved at: {output_stack}")


### Extract features at sample points

In [ ]:
import geopandas as gpd
import rasterio
import pandas as pd
import numpy as np
from shapely.geometry import Point

# === INPUT ===
raster_path = r"DATA_DIR\Combined\2025\Rasters\raster_stack_S2_selected30_mosaic2.tif"
training_points_path = r"DATA_DIR\Combined\2024\Vectors\training_points_70.shp"
output_csv = r"DATA_DIR\Combined\2025\Model\training_data_S2_mosaic2.csv"


# === LOAD SHAPEFILE ===
gdf = gpd.read_file(training_points_path)

# === Open Raster ===
with rasterio.open(raster_path) as src:
    # Reproject points to raster CRS
    if gdf.crs != src.crs:
        print(f"🔄 Reprojecting training points from {gdf.crs} to raster CRS {src.crs}")
        gdf = gdf.to_crs(src.crs)

    band_count = src.count
    band_names = [src.descriptions[i] if src.descriptions[i] else f"Band_{i+1}" for i in range(band_count)]

    features = []
    invalid_indices = []

    for idx, geom in enumerate(gdf.geometry):
        x, y = geom.x, geom.y

        try:
            # Use raster.index safely
            row, col = src.index(x, y)

            # Only sample if inside raster grid
            if 0 <= row < src.height and 0 <= col < src.width:
                values = src.read(window=((row, row+1), (col, col+1))).reshape(band_count)
                features.append(list(values) + [gdf.iloc[idx]["type"]])
            else:
                invalid_indices.append(idx)
        except:
            invalid_indices.append(idx)

# === Save CSV ===
df = pd.DataFrame(features, columns=band_names + ["type"])
df.to_csv(output_csv, index=False)
print(f"✅ Saved extracted features to: {output_csv} ({len(df)} valid points)")


### Merge the two mosaics' training tables (QA, NDVI filter, imputation)

In [ ]:
import pandas as pd
import numpy as np

# === Input file paths ===
mosaic1_path = r"DATA_DIR\Combined\2025\Model\training_data_S2_mosaic1.csv"
mosaic2_path = r"DATA_DIR\Combined\2025\Model\training_data_S2_mosaic2.csv"
output_merged_path = r"DATA_DIR\Combined\2025\Model\training_data_S2_merged.csv"

# === Config ===
invalid_threshold = -1e20
ndvi_col = "Sentinel2_NDVI_Stack_2024_Filled_B5"  # will auto-detect if missing
low_ndvi_threshold = 0.1
keep_n_features = 30  # we will include NDVI in the 30

# === Load CSV files ===
df1 = pd.read_csv(mosaic1_path)
df2 = pd.read_csv(mosaic2_path)

# === Replace extreme values with NaN (numeric only, keep 'type' untouched) ===
for df in (df1, df2):
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    for col in num_cols:
        if col != 'type':
            df[col] = df[col].where(df[col] > invalid_threshold, np.nan)

print(f"🔍 Mosaic 1 NaNs before filling:\n{df1.isna().sum()[df1.isna().sum() > 0]}")
print(f"🔍 Mosaic 2 NaNs before filling:\n{df2.isna().sum()[df2.isna().sum() > 0]}")

# === Find NDVI column robustly if the exact name isn't present ===
if ndvi_col not in df1.columns or ndvi_col not in df2.columns:
    candidates1 = [c for c in df1.columns if "NDVI" in c.upper()]
    candidates2 = [c for c in df2.columns if "NDVI" in c.upper()]
    ndvi_guess = list(set(candidates1).intersection(candidates2))
    if len(ndvi_guess) != 1:
        raise ValueError(f"❌ Could not uniquely identify NDVI column. Found in mosaic1: {candidates1}, mosaic2: {candidates2}")
    ndvi_col = ndvi_guess[0]
    print(f"ℹ️ Using detected NDVI column: {ndvi_col}")

# === Ensure column consistency (preserve Mosaic 1 order) ===
common_columns_ordered = [c for c in df1.columns if c in df2.columns]
if 'type' not in common_columns_ordered:
    raise ValueError("❌ Missing 'type' column in one of the files.")
if ndvi_col not in common_columns_ordered:
    raise ValueError(f"❌ NDVI column '{ndvi_col}' not found in both inputs.")

# === Build feature list: include NDVI this time so total features = 30 ===
feature_candidates = [c for c in common_columns_ordered if c != 'type']  # keep order from mosaic1
# Keep only numeric features (type might be numeric, but we already excluded it)
numeric_in_both = [c for c in feature_candidates
                   if (pd.api.types.is_numeric_dtype(df1[c]) and pd.api.types.is_numeric_dtype(df2[c]))]

feature_columns = numeric_in_both[:keep_n_features]
if len(feature_columns) < keep_n_features:
    print(f"⚠️ Only {len(feature_columns)} numeric common features found; expected {keep_n_features}.")

# Sanity: make sure NDVI is among the selected 30
if ndvi_col not in feature_columns:
    # If we sliced it out, put it back by ensuring it's included
    if ndvi_col in numeric_in_both:
        # Insert NDVI at its original position and trim to length
        if ndvi_col not in feature_columns:
            # find its original position
            pos = numeric_in_both.index(ndvi_col)
            # recreate with NDVI guaranteed in
            feature_columns = (numeric_in_both[:pos+1] + 
                               [c for c in numeric_in_both[pos+1:] if c != ndvi_col])
            feature_columns = feature_columns[:keep_n_features]
    else:
        raise ValueError(f"❌ NDVI column '{ndvi_col}' is not numeric in both dataframes.")

# Final import columns = 30 features + type (NDVI stays in the 30)
final_import_cols = feature_columns + ['type']

df1 = df1[[c for c in final_import_cols if c in df1.columns]].copy()
df2 = df2[[c for c in final_import_cols if c in df2.columns]].copy()

# === Merge first, THEN impute (shared means across mosaics) ===
df_merged = pd.concat([df1, df2], ignore_index=True)

# ✅ STEP 1: Remove invalid slope/aspect (if present)
before_filter = len(df_merged)
if "Slope_resampled" in df_merged.columns and "Aspect_resampled" in df_merged.columns:
    df_merged = df_merged[
        df_merged["Slope_resampled"].between(0, 90) &
        df_merged["Aspect_resampled"].between(0, 360)
    ]
removed = before_filter - len(df_merged)
print(f"\n🧹 Removed {removed} rows with invalid slope/aspect values")

# ✅ STEP 2: NDVI-based QA (report + drop for type==1)
df_merged[ndvi_col] = pd.to_numeric(df_merged[ndvi_col], errors="coerce")

low_ndvi_mask = df_merged[ndvi_col] < low_ndvi_threshold
low_ndvi = df_merged.loc[low_ndvi_mask]
print(f"\n⚠️ Low NDVI (<{low_ndvi_threshold}) points: {len(low_ndvi)} / {len(df_merged)}")
print("🔍 NDVI < threshold distribution by 'type':")
print(low_ndvi['type'].value_counts())

drop_mask = (df_merged['type'] == 1) & (df_merged[ndvi_col] < low_ndvi_threshold)
dropped_rows = int(drop_mask.sum())
df_merged = df_merged.loc[~drop_mask].copy()
print(f"\n🧽 Dropped {dropped_rows} rows where type==1 & {ndvi_col} < {low_ndvi_threshold}")

# ✅ STEP 3: Impute NaNs now (shared column means across both mosaics)
impute_means = df_merged[feature_columns].mean(numeric_only=True)
df_merged[feature_columns] = df_merged[feature_columns].fillna(impute_means)

# === Save merged dataset (30 features incl. NDVI + type) ===
df_merged = df_merged[feature_columns + ['type']]
df_merged.to_csv(output_merged_path, index=False)

print(f"\n📦 Merged dataset saved: {output_merged_path}")
print(f"🧮 Final merged shape: {df_merged.shape}")

# === Summary statistics ===
print("\n📊 Summary Statistics (features only):")
print(df_merged[feature_columns].describe().T[['mean', 'std', 'min', 'max']])

# === Class balance ===
print("\n🧮 Class Distribution:")
print(df_merged['type'].value_counts())


### Train the Random Forest on the selected 30-feature set
Reads the final feature list produced by `04_feature_selection_leakage_free.ipynb` (`selected_features_final.csv`) and trains the deployed model. Hyperparameters are listed in the manuscript parameter table.

In [ ]:
import pandas as pd, joblib
from sklearn.ensemble import RandomForestClassifier

DATA_DIR = "DATA_DIR"  # <-- set project root

training_csv  = f"{DATA_DIR}/Model/training_data_S2_merged.csv"
features_csv  = f"{DATA_DIR}/Model/selected_features_final.csv"   # from notebook 04
model_output  = f"{DATA_DIR}/Model/rf_pinus_mugo.pkl"

data = pd.read_csv(training_csv)
selected = pd.read_csv(features_csv)["feature"].tolist()

X = data[selected]
y = data["type"]

# Deployed classifier. Selection and reported accuracy are handled in notebooks 04 & 05;
# this fit only produces the model used to generate the yearly rasters.
rf = RandomForestClassifier(
    n_estimators=500, max_depth=None, min_samples_split=2,
    min_samples_leaf=1, random_state=42, n_jobs=-1
)
rf.fit(X, y)
joblib.dump(rf, model_output)
print("Saved deployed model ->", model_output)


### Classify each mosaic with the trained RF

In [ ]:
import rasterio
import numpy as np
import joblib
import warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")

# Define paths
raster_stack_path = r"DATA_DIR\Combined\2025\Rasters\raster_stack_S2_selected30_mosaic2.tif"
model_path = r"DATA_DIR\Combined\2025\Model\training_data_S2_merged.pkl"
output_raster = r"DATA_DIR\Combined\2025\Results\Pinus_mugo_classification_selected30_mosaic2.tif"


# Load trained model
rf_model = joblib.load(model_path)
print("✅ Loaded trained Random Forest model.")

# Expected band order from model
expected_band_names = list(rf_model.feature_names_in_)

# Open raster stack
with rasterio.open(raster_stack_path) as src:
    meta = src.meta.copy()
    band_count = src.count
    height, width = src.height, src.width

    # Map raster band descriptions to indices
    raster_band_names = list(src.descriptions)
    name_to_index = {desc: i for i, desc in enumerate(raster_band_names)}

    # Check and warn for mismatches
    mismatched = [b for b in expected_band_names if b not in name_to_index]
    if mismatched:
        raise ValueError(f"❌ Missing bands in raster: {mismatched}")

    # Get band indices in model-required order
    ordered_indices = [name_to_index[name] for name in expected_band_names]

    # Update metadata for output raster
    meta.update({
        "count": 1,
        "dtype": "uint8",
        "nodata": 0,
        "compress": "lzw",
        "BIGTIFF": "YES",
        "tiled": True
    })

    # Tile size for memory-efficient processing
    tile_size = 512

    with rasterio.open(output_raster, "w", **meta) as dst:
        for i in range(0, height, tile_size):
            for j in range(0, width, tile_size):
                window = rasterio.windows.Window(
                    j, i, min(tile_size, width - j), min(tile_size, height - i)
                )

                # Read and reorder bands based on expected model input
                raster_data = np.stack([
                    src.read(b + 1, window=window) for b in ordered_indices
                ]).astype(np.float32)  # Shape: (bands, H, W)

                tile_height, tile_width = raster_data.shape[1], raster_data.shape[2]
                reshaped = raster_data.reshape(len(ordered_indices), -1).T  # (N_pixels, N_bands)

                # Predict only on valid pixels
                invalid_mask = ~np.isfinite(reshaped).all(axis=1)
                predictions = np.zeros((reshaped.shape[0],), dtype=np.uint8)

                if (~invalid_mask).sum() > 0:
                    valid_data = reshaped[~invalid_mask]
                    predictions[~invalid_mask] = rf_model.predict(valid_data).astype(np.uint8)

                classified_tile = predictions.reshape(tile_height, tile_width)
                dst.write(classified_tile, 1, window=window)

print(f"✅ Classification map saved: {output_raster}")


### Merge classified mosaics into the annual map

In [ ]:
import rasterio
from rasterio.merge import merge
import os

# 📂 Input raster file paths
raster1 = r"DATA_DIR\Combined\2025\Results\Pinus_mugo_classification_selected30_mosaic1.tif"
raster2 = r"DATA_DIR\Combined\2025\Results\Pinus_mugo_classification_selected30_mosaic2.tif"
output_path = r"DATA_DIR\Combined\2025\Results\Pinus_mugo_classification_2025.tif"

# 📌 Open raster files
src1 = rasterio.open(raster1)
src2 = rasterio.open(raster2)

# 🔁 Merge using max method (prioritizes higher values like class 1)
merged_array, merged_transform = merge([src1, src2], method='max')

# 📦 Update metadata
out_meta = src1.meta.copy()
out_meta.update({
    "height": merged_array.shape[1],
    "width": merged_array.shape[2],
    "transform": merged_transform,
    "count": merged_array.shape[0],
    "dtype": "uint8",
    "compress": "lzw",
    "BIGTIFF": "YES"
})

# 💾 Write merged raster
with rasterio.open(output_path, "w", **out_meta) as dest:
    dest.write(merged_array.astype("uint8"))

print(f"✅ Merged and saved to: {output_path}")


### Validate a single annual map against independent validation points

In [ ]:
import geopandas as gpd
import rasterio
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Define paths
validation_points_path = r"DATA_DIR\Combined\2024\Vectors\validation_points.shp"# Validation points
classification_map = r"DATA_DIR\Combined\Combined\Results\Classification\merged\2021_merged.tif"   # Predicted map
output_validation_csv = r"DATA_DIR\test_small\Case_4\Pinus_mugo_classification_S2_Topo_CH_RF_validation.csv"

# Load validation points
validation_points = gpd.read_file(validation_points_path)

# Open classified raster
with rasterio.open(classification_map) as src:
    validation_predictions = []
    
    for geom in validation_points.geometry:
        if geom is None:  # Ensure the geometry exists
            validation_predictions.append(np.nan)
            continue

        x, y = geom.x, geom.y  # Get point coordinates
        try:
            row, col = src.index(x, y)  # Convert to raster indices
            value = src.read(1)[row, col]  # Get classification value (0 = Absence, 1 = Presence)
            validation_predictions.append(value)
        except (IndexError, TypeError):  # Handle points outside raster bounds or invalid geometries
            validation_predictions.append(np.nan)

# Add predicted values to validation dataset
validation_points["predicted"] = validation_predictions

# Drop NaN values (if any points were out of raster bounds)
validation_points_clean = validation_points.dropna(subset=["predicted"])

# Save validation results without geometry (to avoid errors with CSV format)
#validation_points_clean[["type", "predicted"]].to_csv(output_validation_csv, index=False)
#print(f"Validation data saved at: {output_validation_csv}")

# Evaluate classification accuracy
y_true = validation_points_clean["type"].astype(int)  # Ground truth
y_pred = validation_points_clean["predicted"].astype(int)  # Model predictions

accuracy = accuracy_score(y_true, y_pred)
conf_matrix = confusion_matrix(y_true, y_pred)
class_report = classification_report(y_true, y_pred)

print(f"Classification Map Validation Accuracy: {accuracy:.4f}")
print("Confusion Matrix:")
print(conf_matrix)
print("Classification Report:")
print(class_report)


### Batch validation across years

In [ ]:
import geopandas as gpd
import rasterio
import numpy as np
import pandas as pd
import os
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# === USER PATHS ===
validation_points_path = r"DATA_DIR\Combined\2024\Vectors\validation_points.shp"
raster_folder = r"DATA_DIR\Combined\Individual"
output_excel = r"DATA_DIR\Combined\Validation\yearly_validation_summary_2025.xlsx"

# === Step 1: Load validation points ===
validation_points = gpd.read_file(validation_points_path)

# === Step 2: Initialize summary list ===
results = []

# === Step 3: Loop over years ===
for year in range(2025, 2026):  # inclusive of 2024
    raster_filename = f"Pinus_mugo_classification_{year}.tif"
    raster_path = os.path.join(raster_folder, raster_filename)
    
    if not os.path.exists(raster_path):
        print(f"❌ Raster not found for {year}: {raster_path}")
        continue

    print(f"🔄 Processing year {year}...")

    # Open raster
    with rasterio.open(raster_path) as src:
        preds = []
        for geom in validation_points.geometry:
            if geom is None:
                preds.append(np.nan)
                continue
            try:
                row, col = src.index(geom.x, geom.y)
                val = src.read(1)[row, col]
                preds.append(val)
            except:
                preds.append(np.nan)
    
    # Add predictions
    validation_points["predicted"] = preds
    valid = validation_points.dropna(subset=["predicted"]).copy()
    valid["predicted"] = valid["predicted"].astype(int)
    valid["type"] = valid["type"].astype(int)
    
    # Compute metrics
    y_true = valid["type"]
    y_pred = valid["predicted"]

    acc = accuracy_score(y_true, y_pred)
    conf_mat = confusion_matrix(y_true, y_pred).tolist()  # make JSON-safe
    report = classification_report(y_true, y_pred, output_dict=True)
    
    # Append results
    results.append({
        "Year": year,
        "Accuracy": acc,
        "Support_0": report["0"]["support"],
        "Support_1": report["1"]["support"],
        "Precision_0": report["0"]["precision"],
        "Precision_1": report["1"]["precision"],
        "Recall_0": report["0"]["recall"],
        "Recall_1": report["1"]["recall"],
        "F1_0": report["0"]["f1-score"],
        "F1_1": report["1"]["f1-score"],
        "Confusion_Matrix": conf_mat,
        "Valid_Points": len(valid)
    })

# === Step 4: Save as Excel ===
df = pd.DataFrame(results)
df.to_excel(output_excel, index=False)
print(f"\n✅ Saved yearly accuracy results to:\n{output_excel}")
